**Group information**

| Family name | First name | Email address |
| ----------- | ---------- | ------------- |
|             |            |               |
|             |            |               |
|             |            |               |

# Spatial and image data: Practice

This tutorial explores fundamental spatial and image processing operations using the `geopandas`, `rasterio` and scikit-image (`skimage`) libraries.

In [ ]:
# Google Colab
# !pip install geopandas rasterio rasterstats

In [ ]:
# Packages
import geopandas as gpd
import numpy as np
import os
import rasterio
import rasterstats
import shutil

from matplotlib import pyplot as plt
from skimage import color, exposure, feature, filters, io, measure, morphology, segmentation, transform
from sklearn import feature_extraction
from urllib import request

In [ ]:
# Utilities

def download_data() -> None:
    '''Downloads data folder'''
    if os.getcwd().endswith('/data'):
        print('Data folder already exists')
    else:
        request.urlretrieve('https://www.dropbox.com/scl/fo/iwdixca5pd1ab3o7tdply/AEu7ThpqVeIFpgF5nvXCkeY?rlkey=d49w1mukq4km5m67fdwe3us6r&dl=1', 'data.zip')
        shutil.unpack_archive('data.zip', 'data')
        os.remove('data.zip')
        os.chdir('data')

def plot_geopandas(gdf:gpd.GeoDataFrame, column:str=None, title:str='', cmap:str='magma', markersize:int=1) -> None:
    '''Plots a GeoDataFrame'''
    fig, ax = plt.subplots(figsize=(5, 5))
    gdf.plot(column=column, cmap=cmap, markersize=markersize, legend=True, ax=ax)
    ax.set_title(title, fontsize=15)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_image(image:np.ndarray, title:str='', cmap:str='gray', figsize=(5, 5)) -> None:
    fig, ax = plt.subplots(1, figsize=figsize)
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=15)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_images(images:list, titles:list=['Image'], cmaps:list=['gray']) -> None:
    nimage = len(images)
    if len(titles) == 1: titles = titles * nimage
    if len(cmaps) == 1:  cmaps  = cmaps  * nimage
    fig, axs = plt.subplots(nrows=1, ncols=nimage, figsize=(10, 10 * nimage))
    for ax, image, title, cmap in zip(axs.ravel(), images, titles, cmaps):
        ax.imshow(image, cmap=cmap)
        ax.set_title(title, fontsize=15)
        ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_histogram(image: np.ndarray, title:str='', color:str='lightblue') -> None:
    fig, ax = plt.subplots(1, figsize=(5, 5))
    ax.hist(image.ravel(), bins=256, density=True, color=color, width=1, edgecolor='black', linewidth=0.1)
    ax.set_title(title, fontsize=20)
    ax.set_xlim(0, 255)
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
# Execute on first run
download_data()

**1. Vector data:** Load the `iron_mines.gpkg` and `un_countries.gpkg` datasets using the `gpd.read_file` function. Plot them using the provided `plot_geopandas` function.

**2. Spatial joins:** Inspect the CRS of both datasets using the `crs` attribute. Reproject the `countries`dataset to match the CRS of the `mines` dataset layer using the `to_crs` method. Perform a spatial join between the `mines` and the `countries` dataset using the `sjoin` method. Filter the result to keep only operating mines that fall within India.

**3. Zonal statistics:** Inspect the CRS of the `mines` dataset and verify the unit of measurement Create a buffer of 10 km around each mine using the `buffer` method. Reproject the `mines` dataset to match the CRS of the raster `population_india_2019.tif` and save the buffered geometries to disk with the `to_file` method. Use `rasterstats.zonal_stats` to compute the total population within each buffer. Store the results as a new column (`population`) in the `mines` dataset.

**4. Images:** Load `powder.jpg` as a `numpy.ndarray` using `io.imread` and plot the image using the provided `plot_image` function. Check the dimensions and the data type of the image using the `shape` and `dtype` attributes.

**5. Feature maps:** Using array indexing, plot the red, green and blue feature maps separately using the provided `plot_images` function. 

**6. Manipulating spatial dimensions:** Load `park_guell.jpg` and plot the image. Crop the image using numpy indexing. Resize and rescale the image using `transform.resize` and `transform.rescale` functions. Check out other image transformations e.g. `transform.rotate`, `transform.flip`.

**7. Colour spaces:** Load `yellowstone.jpg` and plot the image. Convert the image to greyscale using `color.rgb2gray` and plot the result. Convert the image to HSV using `color.rgb2hsv` and plot the resulting channels.

**8. Contrast:** Load `hail.jpg` and plot the image. Stretch the intensities using `exposure.rescale_intensity` using the 2nd and 98th percentile and plot the result. Plot the histogram of the original and the stretched image using the provided `histogram` function.

**9. Edge features:**. Load `madrid.jpg`, plot the image and convert it to greyscale. Compute and plot the horizontal and vertical gradients of the image using `filters.sobel_h` and `filters.sobel_v`. Compute and plot the Sobel edge detection operator using `filters.sobel`.

**10. Texture features:** Load `bocaraton.jpg`, plot the image and convert it to greyscale. Compute the local binary patterns of the image using `feature.local_binary_pattern` using a radius of 1, 2 and 3 and 8, 16 and 24 neighbours respectively.

**11. Segmentation:** Load `almeria.jpg` and plot the image. Compute the Quickshift segmentation using `segmentation.quickshift`. Plot the segments using `segmentation.mark_boundaries`. Repeat the exercise with Slic segmentation using `segmentation.slic`

**12. Morphology:** Load `hail.jpg` and plot the image. Convert it to greyscale, then create a binary image using a threshold. Apply a morphological operation (e.g. `morphology.opening`) to clean up the binary image. Use `measure.label` to identify the distinct clusters.

**BONUS (intermediate)**: Show that convolution can be expressed as a matrix product between a reshaped image matrix (`feature_extraction.image.extract_patches_2d`) and a vector kernel. How does this relate to neural network units?

**BONUS (Advanced)**: Learn one of the Sobel operators from a randomly initialised vector of weights $W$ by minimising the mean square error (MSE) between the convolution output and that of the previous question. Conclude on how convolutional networks can learn image features.

\begin{align*}
    \text{MSE loss: } & L(W) = \frac{1}{n}\Vert\hat{Y} - Y\Vert^2 \\
    \text{Associated gradient: } & \nabla_W L = \frac{2}{n} X^\top (\hat{Y} - Y)
\end{align*}